In [0]:
# Databricks notebook source
# Proyecto Final ElectroCasa
# Notebook: 00_setup
# Objetivo: Configuración inicial de Unity Catalog

CATALOGO = "proyecto_final"

print("====================================")
print(" PROYECTO FINAL - ELECTROCASA")
print("====================================")
print(f"Catálogo seleccionado: {CATALOGO}")

# Verificar la existencia del catálogo
spark.sql(f"DESCRIBE CATALOG {CATALOGO}").show(truncate=False)

# Mostrar los schemas existentes
spark.sql(f"SHOW SCHEMAS IN {CATALOGO}").show(truncate=False)

In [0]:
# ============================================================
# 02. CREACIÓN DE SCHEMAS - ARQUITECTURA MEDALLION
# Proyecto: ElectroCasa
# ============================================================

SCHEMAS = ["bronze", "silver", "gold"]

for esquema in SCHEMAS:

    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS
        {CATALOGO}.{esquema}
    """)

    print(f"Schema creado o existente: {CATALOGO}.{esquema}")

# Verificar los schemas
print("\nSchemas disponibles:")

spark.sql(f"""
    SHOW SCHEMAS IN {CATALOGO}
""").show(truncate=False)

In [0]:
# ============================================================
# 03. CREACIÓN DEL VOLUME DE LANDING
# Proyecto: ElectroCasa
# ============================================================

VOLUME = "vol_landing"

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOGO}.default.{VOLUME}
""")

RUTA_LANDING = f"/Volumes/{CATALOGO}/default/{VOLUME}"

print(f"Volume creado: {CATALOGO}.default.{VOLUME}")
print(f"Ruta de landing: {RUTA_LANDING}")

# Verificar el Volume
spark.sql(f"""
    SHOW VOLUMES IN {CATALOGO}.default
""").show(truncate=False)

In [0]:
# ============================================================
# 04. CREACIÓN DE CARPETAS DE LANDING
# Proyecto: ElectroCasa
# ============================================================

FUENTES = [
    "ventas",
    "catalogo",
    "empleados",
    "resenas",
    "devoluciones",
    "sucursales"
]

for fuente in FUENTES:
    ruta = f"{RUTA_LANDING}/{fuente}"

    dbutils.fs.mkdirs(ruta)

    print(f"Carpeta creada: {ruta}")

# Verificar las carpetas
print("\nContenido del Volume:")

display(dbutils.fs.ls(RUTA_LANDING))

In [0]:
# ============================================================
# 05. VERIFICACIÓN DE GRUPO DE INGENIERÍA
# Proyecto: ElectroCasa
# ============================================================

spark.sql("""
    SELECT
        current_user() AS usuario_actual,
        is_account_group_member(
            'electrocasa_ingenieria'
        ) AS pertenece_ingenieria
""").show(truncate=False)

In [0]:
# ============================================================
# 06. CONFIGURACIÓN DE PERMISOS - UNITY CATALOG
# Proyecto: ElectroCasa
# ============================================================

GRUPO_ING = "`electrocasa_ingenieria`"
GRUPO_ANA = "`electrocasa_analistas`"
GRUPO_AUD = "`electrocasa_auditoria`"

# 1. Permitir el uso del catálogo a los tres grupos
for grupo in [GRUPO_ING, GRUPO_ANA, GRUPO_AUD]:

    spark.sql(f"""
        GRANT USE CATALOG
        ON CATALOG {CATALOGO}
        TO {grupo}
    """)

    print(f"USE CATALOG concedido a {grupo}")


# 2. Ingeniería: acceso a Bronze, Silver y Gold
for esquema in ["bronze", "silver", "gold"]:

    spark.sql(f"""
        GRANT USE SCHEMA, CREATE TABLE,
              CREATE MATERIALIZED VIEW, SELECT, MODIFY
        ON SCHEMA {CATALOGO}.{esquema}
        TO {GRUPO_ING}
    """)

    print(f"Permisos de ingeniería concedidos sobre {esquema}")


# 3. Analistas: únicamente lectura sobre Gold
spark.sql(f"""
    GRANT USE SCHEMA, SELECT
    ON SCHEMA {CATALOGO}.gold
    TO {GRUPO_ANA}
""")

print("Permisos de Analistas concedidos")


# 4. Auditoría: únicamente lectura sobre Gold
spark.sql(f"""
    GRANT USE SCHEMA, SELECT
    ON SCHEMA {CATALOGO}.gold
    TO {GRUPO_AUD}
""")

print("Permisos de Auditoría concedidos")


# 5. Ingeniería: acceso al Volume de landing
spark.sql(f"""
    GRANT USE SCHEMA
    ON SCHEMA {CATALOGO}.default
    TO {GRUPO_ING}
""")

spark.sql(f"""
    GRANT READ VOLUME, WRITE VOLUME
    ON VOLUME {CATALOGO}.default.vol_landing
    TO {GRUPO_ING}
""")

print("Permisos del Volume concedidos a Ingeniería")

In [0]:
# ============================================================
# 07. VERIFICACIÓN DE PERMISOS - UNITY CATALOG
# Proyecto: ElectroCasa
# ============================================================

print("===== PERMISOS DEL CATÁLOGO =====")

display(
    spark.sql(f"""
        SHOW GRANTS ON CATALOG {CATALOGO}
    """)
)

print("===== PERMISOS DE BRONZE =====")

display(
    spark.sql(f"""
        SHOW GRANTS ON SCHEMA {CATALOGO}.bronze
    """)
)

print("===== PERMISOS DE SILVER =====")

display(
    spark.sql(f"""
        SHOW GRANTS ON SCHEMA {CATALOGO}.silver
    """)
)

print("===== PERMISOS DE GOLD =====")

display(
    spark.sql(f"""
        SHOW GRANTS ON SCHEMA {CATALOGO}.gold
    """)
)

print("===== PERMISOS DEL VOLUME =====")

display(
    spark.sql(f"""
        SHOW GRANTS ON VOLUME {CATALOGO}.default.vol_landing
    """)
)